# Agentic RAG Demo Notebook

This notebook demonstrates the final local Agentic RAG prototype built over a policy manual PDF.

It is designed for presentation, not training. The goal is to show the major pipeline units, the retrieval strategy, the LangGraph orchestration logic, and the current system limitations.

## 1. Assignment Mapping

This notebook addresses the main requirements from the assignment:

- local Python implementation
- LangGraph-based orchestration
- RAG over an external text source
- agentic behavior through routing, query rewriting, broader retry search, answer verification, and smalltalk handling
- structured explanation of the architecture, technical choices, bottlenecks, and next steps

In [ ]:
from pathlib import Path
import json
import sys

PROJECT_ROOT = Path.cwd()
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from rag.config import DATA_DIR, SOURCE_DOCUMENT, EMBEDDING_MODEL_NAME, GENERATION_MODEL_NAME, RERANKER_MODEL_NAME
from rag.retrieve import load_retrieval_model, load_reranker, retrieve_chunks, format_results, filter_results
from rag.graph import build_graph

print("Project root:", PROJECT_ROOT)
print("Data dir:", DATA_DIR)
print("Source document:", SOURCE_DOCUMENT.name)
print("Embedding model:", EMBEDDING_MODEL_NAME)
print("Generation model:", GENERATION_MODEL_NAME)
print("Reranker:", RERANKER_MODEL_NAME)

## 2. Data and Pipeline Outputs

The project uses a single policy manual PDF as the knowledge source. The preprocessing pipeline extracts and cleans page text, builds chunks, embeds them, and stores the vector index locally.

In [ ]:
artifacts = {
    "extracted_pages": DATA_DIR / "extracted_pages.json",
    "cleaned_pages": DATA_DIR / "cleaned_pages.json",
    "chunked_documents": DATA_DIR / "chunked_documents.json",
    "embedded_chunks": DATA_DIR / "embedded_chunks.json",
    "test_queries": DATA_DIR / "test_queries.json",
}

for name, path in artifacts.items():
    print(f"{name}: {'present' if path.exists() else 'missing'} -> {path.name}")

In [ ]:
with open(DATA_DIR / "extracted_pages.json", "r", encoding="utf-8") as f:
    extracted_pages = json.load(f)

with open(DATA_DIR / "cleaned_pages.json", "r", encoding="utf-8") as f:
    cleaned_pages = json.load(f)

with open(DATA_DIR / "chunked_documents.json", "r", encoding="utf-8") as f:
    chunked_documents = json.load(f)

print("Extracted pages:", len(extracted_pages))
print("Cleaned pages:", len(cleaned_pages))
print("Chunks:", len(chunked_documents))

## 3. Chunking Strategy

The final chunking strategy is not raw PDF block chunking. It uses page text extraction plus heading/list-aware recursive chunking.

Main ideas:

- remove repeated headers and footer noise
- filter obviously irrelevant pages such as title and table-of-contents pages
- preserve heading-like lines and policy lists
- split by semantic-ish structure before applying token limits

In [ ]:
sample_chunk = chunked_documents[0]
sample_chunk

## 4. Retrieval Layer

The retrieval pipeline is hybrid:

- dense retrieval with `e5-base-v2`
- lexical retrieval with BM25
- reciprocal rank fusion
- reranking with a cross-encoder

This was introduced because pure embedding retrieval was too weak for policy headings, repeated legal terms, and exact titles.

In [ ]:
query = "What is the holiday policy?"
retrieval_model = load_retrieval_model()
reranker = load_reranker()
raw_results = retrieve_chunks(query, retrieval_model, top_k=4, use_reranker=True, reranker=reranker)
results = filter_results(format_results(raw_results))
[(item['metadata']['page_number'], item['metadata']['page_chunk_index']) for item in results]

In [ ]:
print(results[0]['text'])

## 5. Agentic Workflow

The orchestration layer uses LangGraph.

Current decision stages:

1. classify the question
2. detect follow-up questions
3. rewrite contextual follow-up queries
4. run primary hybrid retrieval
5. optionally broaden search if retrieval looks weak
6. generate an answer from retrieved context
7. verify the answer and fall back to extractive output if needed
8. short-circuit smalltalk acknowledgements without retrieval

In [ ]:
graph = build_graph()
result = graph.invoke({"question": "What is the holiday policy?", "chat_history": []})
result['answer']

In [ ]:
result['agent_trace']

## 6. Follow-up Behavior

The system also supports short contextual follow-up questions by rewriting them against recent user history.

In [ ]:
history = [
    {"role": "user", "content": "What is the holiday policy?"},
    {"role": "assistant", "content": result['answer']},
]
follow_up = graph.invoke({"question": "And what about vacation?", "chat_history": history})
follow_up['answer']

In [ ]:
follow_up['agent_trace']

## 7. Smalltalk Routing

To demonstrate that the system does not always retrieve blindly, short acknowledgements are routed away from retrieval.

In [ ]:
smalltalk = graph.invoke({"question": "cool", "chat_history": history})
smalltalk['answer'], smalltalk['agent_trace']

## 8. Evaluation Queries

A small fixed evaluation suite is included to sanity-check the retrieval and answering behavior across multiple policy topics.

In [ ]:
with open(DATA_DIR / "test_queries.json", "r", encoding="utf-8") as f:
    test_queries = json.load(f)

test_queries[:8]

## 9. Bottlenecks and Limitations

Current bottlenecks:

- local answer generation is the slowest part of the system
- GPU availability is critical for acceptable response times
- larger local models caused instability on the available hardware
- some questions still require extractive fallback instead of full natural generation

Current limitations:

- the system is optimized for one policy manual
- retrieval is strong but not perfect for repeated concepts across multiple chapters
- the agentic behavior is practical orchestration, not a fully open-ended multi-agent planner

## 10. Next Steps

Planned improvements:

- stronger evaluation with expected-page labels
- cleaner answer formatting for extractive fallback
- more robust local model serving in a dedicated process
- support for multiple policy documents
- stronger answer verification heuristics for ambiguous questions